# Übung 03: Dateiformate & Ein-/Ausgabe

**Übungsnotebook zur Vorbereitung auf LN 1 (Mo. 09.11.2026)** · Cheatsheet: **Kapitel 3 (Dateiformate & Ein-/Ausgabe)** und Kapitel 11 (Checklisten)

| | |
|---|---|
| **Themen** | CSV-Optionen (Trennzeichen, Dezimalkomma, Tausendertrennzeichen, Encoding, dtype), JSON & `json_normalize`, XML, Excel, ZIP, Parquet, Schreiben von Dateien |
| **Format** | wie die Musterprüfungen AP01–AP03: Teil A Konzeptfragen · Teil B Fehleranalyse · Teil C Code schreiben |
| **Zeitbudget** | ca. 200 Minuten (Prüfungstempo: ca. 4.5 Min. pro Punkt) |
| **Lösungen** | keine – alles ist mit dem Cheatsheet bzw. den Kursunterlagen lösbar. Hinweise wie *(→ 4.5)* verweisen auf den Cheatsheet-Abschnitt. |

**Szenario:** Das fiktive *Energiemanagement Region Winterthur* sammelt monatliche Zählerstände der öffentlichen Liegenschaften. Die Daten stammen aus verschiedenen Systemen: ein CSV-Export aus einer alten Windows-Software, Stammdaten als JSON aus einer API, Tarife als XML und Budgets in Excel.

**So üben Sie prüfungsnah**
- Copilot/KI-Autocomplete ausschalten, wie in der Prüfung (Cheatsheet 1.1).
- Begründungen immer **mit konkretem Bezug zu den Daten** (Spaltennamen, Werte, Zahlen aus dem Output) – sonst gibt es in der Prüfung 0 Punkte.
- Verändern Sie die Ausgangsdaten nicht, sondern arbeiten Sie pro Aufgabe mit eigenen Variablen (z.B. `df_B1`, `df_C2`).
- Am Schluss: *Kernel → Restart & Run All* (Cheatsheet 11.5).

## Setup (wird nicht bewertet)

Die folgende Zelle erzeugt die Übungsdaten **synthetisch und reproduzierbar** (fixer Seed) im Unterordner neben diesem Notebook. Sie brauchen den Code nicht zu verstehen und sollen ihn **nicht verändern** – behandeln Sie die erzeugten Dateien so, als hätten Sie sie in der Prüfung erhalten. *(Natürlich dürfen Sie die Dateien im Editor öffnen und ansehen.)*

In [ ]:
# Libraries
import os
import json
import zipfile
import numpy as np
import pandas as pd

# Settings
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Get current working directory
print(os.getcwd())

In [ ]:
# ---------------------------------------------------------------
# Data generator (do not modify) -> folder 'data_uebung03'
# ---------------------------------------------------------------
import os
import json
import zipfile
import numpy as np
import pandas as pd

rng = np.random.default_rng(303)
D = 'data_uebung03'
os.makedirs(D, exist_ok=True)

objects = [
    ('0007', 'Schulhaus Gütli', 'Winterthur', 1962), ('0012', 'Hallenbad Töss', 'Winterthur', 1974),
    ('0019', 'Stadthaus', 'Winterthur', 1869), ('0042', 'Kindergarten Rosenberg', 'Winterthur', 1995),
    ('0055', 'Werkhof Seen', 'Winterthur', 2025), ('0103', 'Gemeindehaus', 'Seuzach', 1988),
    ('0108', 'Primarschule Birchli', 'Seuzach', 2003), ('0110', 'Mehrzweckhalle', 'Seuzach', 1979),
    ('0201', 'Schulhaus Zentrum', 'Wiesendangen', 1957), ('0205', 'Feuerwehrdepot', 'Wiesendangen', 2011),
    ('0208', 'Alterszentrum Säntisblick', 'Wiesendangen', 1999), ('0301', 'Bibliothek', 'Elsau', 1984),
]
streets = ['Gütlistrasse 3', 'Badweg 10', 'Stadthausstrasse 4', 'Rosenbergstrasse 21', 'Werkstrasse 2',
           'Stationsstrasse 1', 'Birchliweg 7', 'Hallenweg 5', 'Schulstrasse 12', 'Depotstrasse 8',
           'Säntisweg 30', 'Dorfstrasse 15']
plz = {'Winterthur': 8400, 'Seuzach': 8472, 'Wiesendangen': 8542, 'Elsau': 8352}

buildings, meters = [], []
mid = 1
for (oid, name, gem, year), street in zip(objects, streets):
    z = []
    if oid != '0055':                                   # new depot: no meter installed yet
        n_strom = 2 if name in ('Hallenbad Töss', 'Stadthaus', 'Alterszentrum Säntisblick') else 1
        for _ in range(n_strom):
            z.append({'zaehler_id': f'Z{mid:04d}', 'typ': 'Strom', 'installiert': f'20{rng.integers(10, 24)}-0{rng.integers(1, 10)}-01'})
            meters.append((oid, f'Z{mid:04d}', 'Strom', gem)); mid += 1
        if year < 2000:
            z.append({'zaehler_id': f'Z{mid:04d}', 'typ': 'Wärme', 'installiert': f'20{rng.integers(10, 24)}-0{rng.integers(1, 10)}-01'})
            meters.append((oid, f'Z{mid:04d}', 'Wärme', gem)); mid += 1
    buildings.append({'objekt_id': oid, 'name': name, 'gemeinde': gem,
                      'adresse': {'strasse': street, 'plz': plz[gem], 'ort': gem},
                      'baujahr': year, 'zaehler': z})
with open(f'{D}/liegenschaften.json', 'w', encoding='utf-8') as f:
    json.dump(buildings, f, ensure_ascii=False, indent=2)

def swiss_number(x, i):
    """Format like a Swiss Excel export: thousands ', decimal comma, sometimes integer."""
    if i % 3 == 0:
        x = round(x)
        return f'{x:,.0f}'.replace(',', "'")
    return f'{x:,.1f}'.replace(',', "X").replace('.', ',').replace('X', "'")

def readings(year, months):
    rows, i = [], 0
    for oid, zid, typ, gem in meters:
        base = rng.uniform(800, 9000) if typ == 'Strom' else rng.uniform(3000, 25000)
        if oid == '0012':
            base *= 4                                   # indoor pool
        for m in months:
            season = 1 + 0.6 * np.cos((m - 1) / 12 * 2 * np.pi) if typ == 'Wärme' else 1 + 0.1 * np.cos((m - 1) / 12 * 2 * np.pi)
            kwh = base * season * rng.uniform(0.9, 1.1)
            rows.append({'messung_id': f'{year}-{i:05d}', 'objekt_id': oid, 'zaehler_id': zid,
                         'datum': f'{pd.Timestamp(year, m, 1) + pd.offsets.MonthEnd(0):%d.%m.%Y}',
                         'kwh_value': kwh, 'bemerkung': ''})
            i += 1
    df = pd.DataFrame(rows)
    miss = rng.choice(len(df), size=7, replace=False)
    df['kwh'] = [swiss_number(v, j) for j, v in enumerate(df['kwh_value'])]
    df.loc[miss[:4], 'kwh'] = '-'
    df.loc[miss[4:], 'kwh'] = 'n/a'
    df.loc[miss, 'bemerkung'] = 'Zähler defekt; Wert fehlt'
    df.loc[miss, 'kwh_value'] = np.nan
    return df

df26 = readings(2026, range(1, 10))
df26.drop(columns='kwh_value').to_csv(f'{D}/messwerte_2026.csv', sep=';', index=False, encoding='cp1252')
total_2026 = df26['kwh_value'].round(0).where(df26.index % 3 == 0, df26['kwh_value'].round(1)).sum()

df25 = readings(2025, range(1, 13))
df25.drop(columns='kwh_value').to_csv(f'{D}/messwerte_2025.csv', sep=';', index=False, encoding='cp1252')
with zipfile.ZipFile(f'{D}/archiv_2025.zip', 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(f'{D}/messwerte_2025.csv', arcname='messwerte_2025.csv')
os.remove(f'{D}/messwerte_2025.csv')

xml = ['<?xml version="1.0" encoding="UTF-8"?>', '<tarife stand="2026-01-01" waehrung="CHF">']
for gem, s, w in [('Winterthur', 27.4, 13.1), ('Seuzach', 29.8, 14.6), ('Wiesendangen', 31.2, 12.9), ('Elsau', 30.5, 15.0)]:
    xml.append(f'  <tarif gemeinde="{gem}" typ="Strom"><rp_pro_kwh>{s}</rp_pro_kwh><grundgebuehr_chf>120</grundgebuehr_chf></tarif>')
    xml.append(f'  <tarif gemeinde="{gem}" typ="Wärme"><rp_pro_kwh>{w}</rp_pro_kwh><grundgebuehr_chf>240</grundgebuehr_chf></tarif>')
xml.append('</tarife>')
with open(f'{D}/tarife.xml', 'w', encoding='utf-8') as f:
    f.write('\n'.join(xml) + '\n')

with pd.ExcelWriter(f'{D}/budget.xlsx') as xw:
    for yr, fac in [('2025', 1.0), ('2026', 0.95)]:
        b = (df26.dropna().groupby('objekt_id')['kwh_value'].sum() / 9 * 12 * fac).round(-2)
        b.reset_index().rename(columns={'kwh_value': 'budget_kwh'}).to_excel(xw, sheet_name=yr, index=False)

print('Files created:', sorted(os.listdir(D)))
print('Energieabrechnung Jan–Sep 2026, alle Zähler (Kontrollwert):', f'{total_2026:,.1f}'.replace(',', "'"), 'kWh')

### Dateien im Überblick

| Datei | Format | Inhalt |
|---|---|---|
| `messwerte_2026.csv` | CSV (Export einer Windows-Software) | Monatliche Zählerstände Jan–Sep 2026: `messung_id`, `objekt_id`, `zaehler_id`, `datum`, `kwh`, `bemerkung` |
| `liegenschaften.json` | JSON (API-Antwort) | Stammdaten der Liegenschaften inkl. Adresse und Liste der installierten Zähler |
| `tarife.xml` | XML | Energietarife pro Gemeinde und Energieart in Rappen pro kWh |
| `budget.xlsx` | Excel (Sheets `2025`, `2026`) | Verbrauchsbudget pro Liegenschaft in kWh |
| `archiv_2025.zip` | ZIP mit einer CSV-Datei | Messwerte 2025 im gleichen Format wie 2026 |

Öffnen Sie die Dateien zuerst im Editor (CSV und XML als Text, JSON mit Einrückung), bevor Sie mit dem Einlesen beginnen.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnen die Übungsaufgaben — 44 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zu den Dateien des Energiemanagements werden nicht gewertet.*

### A1 (4 Punkte)

Welches Dateiformat (CSV, JSON, Parquet, Excel) wählen Sie für die folgenden drei Situationen? Begründen Sie jeweils mit einer Eigenschaft des Formats. *(→ 3.1)*

1. Die Liegenschaften mit ihrer **variablen Anzahl** Zähler sollen an eine andere Applikation übergeben werden.
2. Die Messwerte aller Gemeinden des Kantons (ca. 50 Mio. Zeilen, 25 Spalten) sollen archiviert werden; Analysen lesen meist nur 2–3 Spalten.
3. Der Hauswart möchte die Budgetzahlen pro Liegenschaft mit seiner Tabellenkalkulation bearbeiten.

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

Sehen Sie sich in `liegenschaften.json` die Einträge `0012` (Hallenbad Töss) und `0055` (Werkhof Seen) an.

a) In welche Python-Datentypen werden `{...}`, `[...]`, die Zahl `1974` und das Feld `"zaehler": []` beim Einlesen mit `json.load()` übersetzt?

b) Wie viele Zeilen erzeugt `pd.json_normalize(data, record_path='zaehler', meta=['objekt_id'])` **für diese beiden Liegenschaften** zusammen? Begründen Sie, was mit dem Werkhof Seen passiert, und warum das für eine Auswertung «Anzahl Zähler pro Liegenschaft» relevant ist. *(→ 3.3)*

**Ihre Antwort:**

> [Hier eintragen]

### A3 (4 Punkte)

Eine Kollegin liest `messwerte_2026.csv` auf ihrem Mac ein und erhält zwei Auffälligkeiten:

- In der Spalte `bemerkung` steht `ZÃ¤hler defekt` statt `Zähler defekt` (bzw. es erscheint ein `UnicodeDecodeError`).
- Nachdem sie die Datei mit `df.to_csv('kopie.csv')` gespeichert und wieder eingelesen hat, gibt es eine neue Spalte `Unnamed: 0`.

Erklären Sie die Ursache beider Auffälligkeiten und wie man sie vermeidet. *(→ 3.2)*

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – Fehleranalyse Python-Code (16 Punkte)

*Jeder Code läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Ergebnis.*

### B1 (8 Punkte)

**Ziel:** Gesamtverbrauch Jan–Sep 2026 über **alle Zähler** sowie Verbrauch pro Liegenschaft (mit Liegenschaftsname aus `liegenschaften.json`), absteigend sortiert.

Der Kontrollwert der Energieabrechnung wird im Setup ausgegeben. **Alle 11 Liegenschaften mit Zählern** haben Messwerte.

In [ ]:
# Original code (do not modify this cell)

# Read the measurements
df_B1 = pd.read_csv('data_uebung03/messwerte_2026.csv', sep=';', decimal=',', encoding='cp1252')
df_B1['kwh'] = pd.to_numeric(df_B1['kwh'], errors='coerce')
df_B1['objekt_id'] = df_B1['objekt_id'].astype(str)

# Read the master data
with open('data_uebung03/liegenschaften.json', 'r', encoding='utf-8') as f:
    objekte = pd.DataFrame(json.load(f))[['objekt_id', 'name']]

# Consumption per building
df_B1 = df_B1.merge(objekte, on='objekt_id', how='left')
print('Gesamtverbrauch (kWh):', round(df_B1['kwh'].sum(), 1))
print(df_B1.groupby('name')['kwh'].sum().sort_values(ascending=False))

**Aufgabe:** Finden Sie die Ursachen (Tipp: `df.dtypes`, `df.head()`, `value_counts()` der Rohwerte und die Rohdatei im Editor). Schreiben Sie den korrigierten Code (Gerüst unten), sodass der Gesamtverbrauch dem Kontrollwert entspricht und alle 11 Liegenschaften erscheinen. Geben Sie zusätzlich aus, wie viele Messwerte fehlen. Begründen Sie jede Änderung. *(→ 3.2, 6.2, 6.8)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

# Read the measurements

# Read the master data

# Consumption per building

### B2 (8 Punkte)

**Ziel:** Anzahl installierter Zähler pro Gemeinde, getrennt nach Energieart (`Strom`, `Wärme`). **Alle Gemeinden** sollen erscheinen.

In [ ]:
# Original code (do not modify this cell)
with open('data_uebung03/liegenschaften.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df_B2 = pd.DataFrame(data)
df_B2['n_zaehler'] = 1
print(df_B2.groupby('gemeinde')['n_zaehler'].sum())
print('Total Zähler:', df_B2['n_zaehler'].sum())

**Aufgabe:** Erklären Sie, was der Code tatsächlich zählt. Schreiben Sie den korrigierten Code, der eine Tabelle **Gemeinde × Energieart** mit der Anzahl Zähler liefert (fehlende Kombinationen = 0) und das korrekte Total ausgibt. Begründen Sie jede Änderung und wie Sie mit Liegenschaften ohne Zähler umgehen. *(→ 3.3, 6.9)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierter Code:**

In [ ]:
# Korrigierten Code unter den Kommentaren eintragen

---
## Teil C – Python-Code schreiben (16 Punkte)

*Schreiben Sie Python-Code und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen.*

### C1 (4 Punkte)

Lesen Sie `tarife.xml` mit `pd.read_xml()` ein. Berechnen Sie die **Energiekosten Jan–Sep 2026 in CHF** pro Liegenschaft und Energieart:

`kosten_chf = kwh × rp_pro_kwh / 100` (die Grundgebühr ignorieren Sie).

Ausgabe: `name`, `typ`, `kwh`, `kosten_chf` (gerundet auf 2 Stellen), absteigend nach `kosten_chf`. Begründen Sie, über welche Schlüssel Sie die drei Quellen (Messwerte, Liegenschaften/Zähler, Tarife) verbinden. *(→ 3.1, 3.3, 6.8)*

*Hinweis:* Die Energieart eines Messwerts steht nicht in der CSV – nur in `liegenschaften.json` beim jeweiligen Zähler.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C2 (4 Punkte)

Speichern Sie Ihr Ergebnis aus C1 (oder die korrekt eingelesenen Messwerte aus B1, inkl. einer Datumsspalte als `datetime`) **zweimal**:

1. als CSV `data_uebung03/export_c2.csv` mit Semikolon, Dezimalkomma, UTF-8 und **ohne** Index,
2. als Parquet `data_uebung03/export_c2.parquet`.

Lesen Sie beide Dateien wieder ein und vergleichen Sie `dtypes` und `shape` mit dem Original. Begründen Sie die Unterschiede. *(→ 3.1, 3.2)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C3 (4 Punkte)

1. Lesen Sie die Messwerte 2025 **direkt aus** `archiv_2025.zip` ein (ohne die Datei manuell zu entpacken) und kontrollieren Sie mit `zipfile.ZipFile(...).namelist()`, welche Datei(en) das Archiv enthält.
2. Lesen Sie **alle** Sheets von `budget.xlsx` mit **einem** Aufruf ein und kombinieren Sie sie zu einem Data Frame mit einer zusätzlichen Spalte `jahr`.
3. Vergleichen Sie für 2025 pro Liegenschaft den **tatsächlichen** Jahresverbrauch mit dem Budget: Spalten `objekt_id`, `ist_kwh`, `budget_kwh`, `abweichung_pct` (gerundet auf 1 Stelle), absteigend nach `abweichung_pct`.

Begründen Sie, welche Einleseoptionen Sie aus B1 wiederverwenden müssen und warum. *(→ 3.1, 3.2)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

### C4 (4 Punkte)

Erstellen Sie aus `liegenschaften.json` eine **flache Adressliste** mit den Spalten `objekt_id`, `name`, `strasse`, `plz`, `ort` (Tipp: `pd.json_normalize` erzeugt Spaltennamen wie `adresse.strasse` – benennen Sie diese um).

Schreiben Sie die Liste als JSON-Datei `data_uebung03/adressliste.json` im Format «Liste von Objekten» (`orient='records'`), sodass Umlaute **lesbar** gespeichert werden (`Töss`, nicht `T\u00f6ss`). Lesen Sie die ersten 8 Zeilen der Datei als Text ein und geben Sie sie aus, um das Ergebnis zu prüfen. Begründen Sie Ihre Optionen beim Schreiben. *(→ 3.3, 3.4)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Libraries
import os, platform, socket
from platform import python_version
from datetime import datetime

# Print system information
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('-----------------------------------')